In [1]:
import os
import time
import pickle

import pandas as pd
import numpy as np
import torch

import requests
import json
from bs4 import BeautifulSoup

from chembl_webresource_client.new_client import new_client
from bioservices import UniProt
from sentence_transformers import SentenceTransformer

# NB: use conda environement for this
# from rdkit.Chem import AllChem
# from rdkit import Chem

/Users/glouno/sourceCode/DALAS-Project/.venv/lib/python3.11/site-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version
/Users/glouno/sourceCode/DALAS-Project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MESH_URL = "https://www.ncbi.nlm.nih.gov/mesh?Db=mesh&Cmd=DetailsSearch&Term=%22Autoimmune+Diseases%22%5BMeSH+Terms%5D"

# ChEMBL 
molecule = new_client.molecule
drug_indication = new_client.drug_indication
target = new_client.target
mechanism = new_client.mechanism

# activity = new_client.activity # TODO: get data from there as well

# Open Targets 
OT_URL = "https://api.platform.opentargets.org/api/v4/graphql"

# Clinical Trials
NCT_URL = "https://clinicaltrials.gov/api/v2/studies"

# Sentence Transformer
ST_MODEL = "neuml/pubmedbert-base-embeddings"

# Uniprot for ID mapping
up = UniProt()


# MeSH IDs

In [3]:
# Exctract all MeSH terms of autoimmune diseases
# Site with MeSH terms with autoimmune diseases
if os.path.exists("../data/mesh_ids.pkl"):
    with open("../data/mesh_ids.pkl", "rb") as f:
        mesh_ids = pickle.load(f)
else:
    url = MESH_URL
    response = requests.get(url)
    if response.status_code == 200:
        html_content = response.text
        soup = BeautifulSoup(html_content, "html.parser")

    soup.find_all("span", string="Autoimmune Diseases")
    autoimm_ul = soup.find("span", string="Autoimmune Diseases").find_all_next("ul")
    autoimm_ul[8].find_all("a")[0].get("href")

    mesh_ids = []

    for disease_a in autoimm_ul[8].find_all("a"):
        disease_url = disease_a.get('href')
        response = requests.get(f'https://www.ncbi.nlm.nih.gov{disease_url}')
        if response.status_code == 200:
            html_content = response.text
            disease_soup = BeautifulSoup(html_content, "html.parser")
            mesh_ids.append(disease_soup.find("p", string=lambda text: text and text.startswith("MeSH Unique ID:")).text.split()[-1])

# Drugs and Indications

In [4]:
indications_df = pd.DataFrame(drug_indication.filter(mesh_id__in=mesh_ids))
chembl_ids = pd.unique(indications_df["molecule_chembl_id"]).tolist()
drugs_df = pd.DataFrame(molecule.filter(molecule_chembl_id__in = chembl_ids))

# Drug targets

In [5]:
mechanism_df = pd.DataFrame(
    mechanism.filter(
    molecule_chembl_id__in = chembl_ids
    ).only(["molecule_chembl_id",
            "action_type",
            "disease_efficacy",
            "mechanism_of_action", 
            "target_chembl_id"])
)
target_ids = mechanism_df["target_chembl_id"].unique().tolist()
targets_df = pd.DataFrame(
    target.filter(
        target_chembl_id__in = target_ids
        ).only(["target_chembl_id",
                "target_type", 
                "target_components"])
)

targets_df = targets_df[
    targets_df["target_type"].str.contains("PROTEIN") &
    (targets_df["target_components"].apply(len) > 0)
    ]

targets_df["uniprot_ids"] = targets_df["target_components"].apply(
    lambda comps: [ref["xref_id"]
                   for c in comps
                   for ref in c.get("target_component_xrefs", [])
                   if  ref.get("xref_src_db") == "UniProt"]
)

In [6]:
up_id_map = up.mapping(fr="UniProtKB", to="UniProtKB", query=list({ x for ids in targets_df["uniprot_ids"] for x in ids})) # type: ignore

100%|██████████| 110/110 [01:35<00:00,  1.15it/s]


In [7]:
primary_up_ids = list({result["to"]["primaryAccession"] for result in up_id_map["results"]})
targets_df["uniprot_ids"] = targets_df["uniprot_ids"].apply(
    lambda up_ids: [up_id for up_id in up_ids if up_id in primary_up_ids]
)

In [8]:
# TODO: reactome (pretty heavy)
# react_id_map = up.mapping(fr="UniProtKB_AC-ID", to="Reactome", query=list({ x for ids in targets_df["uniprot_ids"] for x in ids})) # type: ignore
# targets_df["reactome_ids"] = targets_df["uniprot_ids"].apply(
#     lambda up_ids: [up_id for up_id in up_ids if up_id in primary_up_ids]
# )

In [9]:
mechanism_df = mechanism_df.merge(targets_df, on="target_chembl_id")

# Disease targets and info

In [10]:
nb_top_targets = 5
ot_targets_info = []
for efo_id in indications_df["efo_id"].unique():
  query_string = """
  query disease($efoId: String!, $size: Int!){
      disease(efoId: $efoId) {
      id
      name
      description

      associatedTargets(page: {index: 0, size: $size}, orderByScore: "score desc"){
      rows{
          target {
              id
              proteinIds{
                  id
                  source
              }
          }
          score
        }
      }

      phenotypes {
        rows {
          phenotypeHPO {
            id
            name
            description
          }
        }
      }

  }
  }
  """

  ot_response = requests.post(OT_URL,
                               json={"query": query_string, 
                                     "variables": {"efoId": efo_id.replace(":","_"), 
                                                   "size" : nb_top_targets}})

  if ot_response.status_code == 200:
    ot_targets_info.append(json.loads(ot_response.text)["data"]["disease"])
  else:
    print(ot_response.status_code)
diseases_df = pd.DataFrame(ot_targets_info)
diseases_df["ensembl_ids"] = diseases_df["associatedTargets"].apply(
    lambda result: [row["target"]["id"]
                    for row in result["rows"]]
)
ensembl_id_map = up.mapping(fr="Ensembl", to="UniProtKB", query=list({ x for ids in diseases_df["ensembl_ids"] for x in ids})) # type: ignore
uniprot_map = {result["from"]:result["to"]["primaryAccession"] for result in ensembl_id_map["results"]}
diseases_df["uniprot_ids"] = diseases_df["ensembl_ids"].apply(
    lambda ensembl_ids: [ uniprot_map[ensembl_id] for ensembl_id in ensembl_ids]
)
# get direction of association for each target

diseases_df["target_evidence"] = [[] for _ in range(diseases_df.shape[0])]
nb_evidences = 100
for i, disease in diseases_df.iterrows():
    query_string = """
    query disease($efoId: String!, $ensemblIds : [String!]!, $size: Int!){
    disease(efoId: $efoId) {
        #id
        evidences(ensemblIds: $ensemblIds, size: $size){ 
        count
        rows{
            target{
            id
            }
            score
            datatypeId
            variantEffect
            targetModulation
            targetRole
            directionOnTrait
        }
        }
    }
    }
    """

    ot_response = requests.post(OT_URL, 
                                json={"query": query_string, 
                                      "variables": {"efoId": disease["id"], 
                                                    "ensemblIds": disease["ensembl_ids"], 
                                                    "size": nb_evidences}})

    if ot_response.status_code == 200:
        diseases_df.loc[i, "target_evidence"].append(json.loads(ot_response.text)["data"]["disease"]["evidences"]["rows"])
    else:
        print(ot_response.status_code)

100%|██████████| 27/27 [00:12<00:00,  2.13it/s]


# Inspect indications

In [11]:
nct_refs = []
indications_df["nct_ids"] = None
for i, indication in indications_df.iterrows():
    for ref in indication["indication_refs"]:
        if ref["ref_type"] == "ClinicalTrials":
            indications_df.at[i,"nct_ids"] = ref["ref_id"].split(",")
            nct_refs.append(ref["ref_id"].split(","))    

all_nct_refs = np.unique([nct_ref for nct_ref_list in nct_refs for nct_ref in nct_ref_list])
if os.path.exists("../data/trials_df.pkl"):
    with open("../data/trials_df.pkl", "rb") as f:
        trials_df = pickle.load(f)
else:
    nct_info = []
    for i in range(0, len(all_nct_refs), 3):
        if i%150==0: print(f"Got {i} out of {len(all_nct_refs)} studies")
        nct_response = requests.get(f"{NCT_URL}?filter.ids={'0%7C'.join(all_nct_refs[i:i+3])}&format=json")
        time.sleep(0.1)  # 0.1 sec pause to respect 10 req/sec
        if nct_response.status_code == 200:
            nct_info.append(nct_response.json()["studies"])
        else:
            print(nct_response.status_code)
    trials_df = pd.DataFrame([ct for cts in nct_info for ct in cts])

# Embeddings

In [12]:
model = SentenceTransformer(ST_MODEL) 

In [13]:
drug_names = drugs_df["pref_name"].str.lower()
disease_names = diseases_df["name"].str.lower()
# Compute cosine similarities
similarities = model.similarity(model.encode(drug_names), model.encode(disease_names))
name_similarities = [ {"disease_id" : disease["id"], "drug_id" : drug["molecule_chembl_id"], "disease_name":disease["name"].lower(), "drug_name":drug["pref_name"].lower(), "name_similarity" : float(similarities[i,j])} for j, disease in diseases_df.iterrows() for i, drug in drugs_df.iterrows()]
embeddings_df = pd.DataFrame(name_similarities)

# Put All Together

Raw data:
- Drug:
    - `drugs_df`
    - `mechanism_df` 

- Disease:
    - `diseases_df` 

- Drug-Disease:
    - `indications_df` 
    - `trials_df` 
    - `embeddings_df`


## Drugs Data 

In [ ]:
# Unpack json, keep interesting variables only
final_drugs_df = (
    drugs_df[
        (drugs_df['availability_type'] > 0)
        & ~drugs_df['withdrawn_flag']
    ]
    .drop(columns=['atc_classifications', 'availability_type',
                   'cross_references','molecule_hierarchy',
                   'molecule_synonyms','polymer_flag',
                   'usan_stem', 'usan_substem'])
)
final_drugs_df["biotherapeutic"] = final_drugs_df["biotherapeutic"].notnull().astype(int)

chirality_dict = {2:"achiral", 1:"single_enantiomer", 0:"mixture", -1:None}
final_drugs_df["chirality"] = final_drugs_df["chirality"].apply(lambda x: chirality_dict[x])
final_drugs_df = pd.concat([final_drugs_df, 
                            pd.DataFrame([d if d is not None else {} 
                                          for d in final_drugs_df["molecule_properties"].to_list()]),
                            pd.DataFrame([d if d is not None else {} 
                                          for d in final_drugs_df["molecule_structures"].to_list()])
                            ], axis=1
                        ).drop(columns=["molecule_properties","molecule_structures","molfile"])
final_drugs_df["pref_name"] = final_drugs_df["pref_name"].str.lower()

first_columns = ['molecule_chembl_id','pref_name']
final_drugs_df = final_drugs_df[first_columns + [c for c in final_drugs_df.columns if c not in first_columns]]
final_drugs_df.rename(columns={
    final_drugs_df.columns[0]: "drug_id",
    final_drugs_df.columns[1]: "drug_name"
}, inplace=True)

# Add to each drug a dictionary {action_type : [targets]} 
final_drugs_df["targets"] = None
for i, drug in final_drugs_df.iterrows():
    drug_targets = mechanism_df.loc[mechanism_df["molecule_chembl_id"]== drug["drug_id"],
                                    ["action_type","uniprot_ids"]]
    if drug_targets.shape[0]>0:
        targets_dict = dict()
        for _,r in drug_targets.iterrows():
            action_type = r["action_type"]
            if action_type not in targets_dict:
                targets_dict[action_type] = []
            targets_dict[r["action_type"]] += r["uniprot_ids"]
        final_drugs_df.at[i,"targets"] = targets_dict

In [ ]:
final_drugs_df

## Disease Data

In [ ]:
diseases_df # TODO

## Drug-Disease Data

TODO: do df with all combinations - maybe restrict some combinations (exculde obviously bad matches)

TODO: don't forget to get reactome pathways and engeneer features based on shared pathways

In [ ]:
# Extract interesting data from clinical trials - dates and results
final_trials_df = trials_df.drop(columns=["annotationSection", "documentSection"]) # type: ignore
final_trials_df["nct_id"] = None 
final_trials_df["status"] = None
final_trials_df["phase"] = None
final_trials_df["success"] = None
final_trials_df["median_p_value"] = None
final_trials_df["p_value_list"] = None
for i, study in final_trials_df.iterrows():
    # TODO: choose better dates
    final_trials_df.loc[i, "nct_id"] = study["protocolSection"]["identificationModule"]["nctId"]
    final_trials_df.loc[i, "status"] = study["protocolSection"]["statusModule"]["overallStatus"]
    final_trials_df.loc[i, "start_date"] = study["protocolSection"]["statusModule"].get("startDateStruct",{"date":None})["date"]
    final_trials_df.loc[i, "end_date"] = study["protocolSection"]["statusModule"].get("completionDateStruct",{"date":None})["date"]
    final_trials_df.loc[i, "why_stopped"] = study["protocolSection"]["statusModule"].get("whyStopped", None)
    phases = study["protocolSection"]["designModule"].get("phases",[])
    phases = [int(phase[-1]) for phase in phases if phase != "NA"]
    if phases:
        final_trials_df.loc[i, "phase"] = np.max(phases)
    # Get p-values
    # if "conditionBrowseModule" in study["derivedSection"]:
    #     found_mesh = [mesh['id'] for mesh in  study["derivedSection"]["conditionBrowseModule"]["meshes"]]
    #     if True in [mesh in mesh_ids for mesh in found_mesh]:
    if study["hasResults"]:
        measures = study["resultsSection"]["outcomeMeasuresModule"]["outcomeMeasures"]
        p_values = [measure["analyses"][0]["pValue"] for measure in measures if "analyses" in measure if "pValue" in measure["analyses"][0]]
        p_values = [float(''.join([ch for ch in p if ch.isdigit() or ch=="."])) for p in p_values]
        if len(p_values) > 0:
            final_trials_df.at[i, "p_value_list"] = p_values
final_trials_df = final_trials_df.drop(columns=["derivedSection","protocolSection"])
first_columns = ['nct_id','success',"median_p_value",'phase','status', 'hasResults','why_stopped']
final_trials_df = final_trials_df[first_columns + [c for c in final_trials_df.columns if c not in first_columns]]

for i, study in final_trials_df.iterrows():
    if study["p_value_list"]:
        medp = np.median(study["p_value_list"])
        prop05 = (np.array(study["p_value_list"]) < 0.05).sum() / len(study["p_value_list"])
        conflicting = (np.min(study["p_value_list"]) < 0.05) and ((np.array(study["p_value_list"]) > 0.2).sum() / len(study["p_value_list"]) > 0.5)
        if medp <= 0.05 or (medp <= 0.1 and prop05 >= 0.5):
            final_trials_df.loc[i, "success"] = "success"
        elif 0.05 < medp <= 0.20 or (0.10 < medp <= 0.50 and 0.1 <= prop05 < 0.5) or conflicting:
            final_trials_df.loc[i, "success"] = "unknown"
        elif medp > 0.2 and prop05 < 0.1:
            final_trials_df.loc[i, "success"] = "fail"

In [ ]:
# Add this trial info to corresponding indications
final_indications_df = indications_df.drop(columns=["drugind_id", "mesh_heading", "parent_molecule_chembl_id"])
final_indications_df["nct_evidence"] = None

for i, indication in final_indications_df.iterrows():
    # TODO: finish logic with success
    # add dates 
    if indication["nct_ids"]:
        success_list = []
        for nct_id in indication["nct_ids"]:
            success = final_trials_df.loc[final_trials_df["nct_id"]==nct_id,"success"].values
            if len(success)>0:
                if success[0] is not None:
                    success_list.append(success[0] + str(final_trials_df.loc[final_trials_df["nct_id"]==nct_id,"phase"].values[0]))
        final_indications_df.at[i, "nct_evidence"] = success_list
final_indications_df.rename(columns={"molecule_chembl_id":"drug_id"}, inplace=True)
first_columns = ['drug_id','efo_term','efo_id','mesh_id', 'max_phase_for_ind','nct_evidence']
final_indications_df= final_indications_df[first_columns + [c for c in final_indications_df.columns if c not in first_columns]]